# 08 — Modelos baseline para classificação de TDAH (ADHD-200)
Objetivo

Treinar e comparar SVM, Random Forest e XGBoost usando as características de conectividade funcional preparadas no Notebook 07.
Protocolo de avaliação

- Utiliza X, y, ids, sites e fold_id de data/processed/dataset.npz.
- Usa exatamente os folds agrupados por centro de aquisição (site) definidos no Notebook 07.
- Em cada rodada, um fold inteiro é reservado para validação; os demais são usados no treinamento.
- Cada participante recebe uma previsão fora do treino (out-of-fold, OOF).
- O SVM usa padronização dentro de um Pipeline, ajustada somente com os dados de treino do fold.
- Os hiperparâmetros são definidos previamente; não há busca automática de hiperparâmetros nesta versão.

Interpretação: os resultados são estimativas de validação cruzada agrupada por centro. Como o Notebook 07 não reservou um conjunto de teste independente, eles não devem ser descritos como resultados de um teste final independente.

Codificação esperada: 0 = controle e 1 = TDAH. O Notebook verifica a codificação carregada antes de treinar.

In [3]:
# 1. Imports, configuração e carregamento do dataset

from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("default")
SEED = 42

# Localizar a raiz do projeto
cwd = Path.cwd().resolve()

raiz_projeto = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "config.py").is_file()
        and (p / "notebooks").is_dir()
    ),
    None
)

if raiz_projeto is None:
    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. "
        "Abra o notebook dentro da pasta notebooks do projeto TCC_TDAH."
    )

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

import config

config.criar_pastas()

# Localizar o dataset gerado pelo Notebook 07
arquivo_dataset = Path(config.ARQ_DATASET)

if not arquivo_dataset.is_file():
    raise FileNotFoundError(
        f"Dataset não encontrado: {arquivo_dataset}. "
        "Execute primeiro o Notebook 07."
    )

# Carregar o dataset
# allow_pickle=True permite ler os arrays de objetos do arquivo.
# Utilize somente porque o arquivo foi gerado pelo seu próprio projeto.
with np.load(arquivo_dataset, allow_pickle=True) as dados:

    obrigatorias = {"X", "y", "ids", "sites", "fold_id"}
    ausentes = obrigatorias - set(dados.files)

    if ausentes:
        raise KeyError(
            f"Chaves ausentes em dataset.npz: {sorted(ausentes)}"
        )

    X = np.asarray(dados["X"], dtype=np.float32)
    y = np.asarray(dados["y"], dtype=np.int64).ravel()
    ids = np.asarray(dados["ids"]).astype(str)
    sites = np.asarray(dados["sites"]).astype(str)
    fold_id = np.asarray(dados["fold_id"], dtype=np.int64).ravel()

# Validar dimensões e conteúdo
if X.ndim != 2:
    raise ValueError(
        f"X deve ser uma matriz 2D. Formato recebido: {X.shape}"
    )

if not (
    len(X) == len(y) == len(ids) == len(sites) == len(fold_id)
):
    raise ValueError(
        "X, y, ids, sites e fold_id têm quantidades diferentes."
    )

if not np.isfinite(X).all():
    raise ValueError("X contém valores NaN ou infinitos.")

if set(np.unique(y).tolist()) != {0, 1}:
    raise ValueError(
        f"Esperadas as classes 0 e 1, mas foram encontradas: "
        f"{np.unique(y).tolist()}"
    )

if np.any(fold_id < 0):
    raise ValueError(
        "Há participantes sem fold definido. "
        "Revise o Notebook 07."
    )

# Exibir informações do dataset
print("Raiz do projeto:", raiz_projeto)
print("Dataset:", arquivo_dataset)
print("Formato de X:", X.shape)
print("Total de participantes:", len(y))
print(
    "Distribuição das classes (0=controle, 1=TDAH):",
    pd.Series(y).value_counts().sort_index().to_dict()
)
print("Centros de aquisição:", len(np.unique(sites)))
print("Folds:", sorted(np.unique(fold_id).tolist()))

Raiz do projeto: /home/maria/TCC_TDAH
Dataset: /home/maria/TCC_TDAH/data/processed/dataset.npz
Formato de X: (709, 17955)
Total de participantes: 709
Distribuição das classes (0=controle, 1=TDAH): {0: 453, 1: 256}
Centros de aquisição: 9
Folds: [0, 1, 2, 3, 4]


In [4]:
# 2. Verificar os folds antes do treinamento
folds = sorted(np.unique(fold_id).tolist())
n_folds_config = getattr(config, "N_FOLDS", len(folds))
if len(folds) != int(n_folds_config):
    raise ValueError(
        f"Foram encontrados {len(folds)} folds, mas config.N_FOLDS={n_folds_config}."
    )

resumo_folds = []
for fold in folds:
    idx_teste = fold_id == fold
    idx_treino = ~idx_teste

    if set(np.unique(y[idx_teste]).tolist()) != {0, 1}:
        raise ValueError(
            f"O fold {fold} não contém as duas classes no conjunto de validação. "
            "Revise os folds no Notebook 07."
        )

    sites_treino = set(sites[idx_treino])
    sites_teste = set(sites[idx_teste])
    sobreposicao = sites_treino.intersection(sites_teste)
    if sobreposicao:
        raise ValueError(
            f"Vazamento de centros no fold {fold}: {sorted(sobreposicao)}"
        )

    resumo_folds.append({
        "fold": int(fold),
        "n_treino": int(idx_treino.sum()),
        "n_validacao": int(idx_teste.sum()),
        "controles_validacao": int((y[idx_teste] == 0).sum()),
        "tdah_validacao": int((y[idx_teste] == 1).sum()),
        "n_centros_teste": len(sites_teste),
        "sobreposicao_centros": len(sobreposicao)
    })

df_folds = pd.DataFrame(resumo_folds)
display(df_folds)
print("Verificação concluída: os centros de treino e validação não se sobrepõem.")


,fold,n_treino,n_validacao,controles_validacao,tdah_validacao,n_centros_teste,sobreposicao_centros
0,0,590,119,85,34,2,0
1,1,587,122,81,41,2,0
2,2,580,129,109,20,2,0
3,3,560,149,91,58,2,0
4,4,519,190,87,103,1,0


Verificação concluída: os centros de treino e validação não se sobrepõem.


In [5]:
# 3. Definir os três modelos
# Os hiperparâmetros são fixados previamente para esta comparação baseline.
# O SVM usa StandardScaler dentro do Pipeline, para evitar vazamento entre folds.
from xgboost import XGBClassifier

SEED = int(getattr(config, "SEED", 42))

modelos = {
    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=1.0,
            kernel="rbf",
            class_weight="balanced",
            probability=True,
            random_state=SEED
        ))
    ]),
    "Random_Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist"
    )
}
print("Modelos:", ", ".join(modelos.keys()))


Modelos: SVM_RBF, Random_Forest, XGBoost


In [6]:
# 4. Treinar e avaliar: previsões out-of-fold (OOF)
# Cada participante é previsto apenas pelo modelo que não foi treinado com seu fold.
pred_oof = {nome: np.full(len(y), -1, dtype=np.int8) for nome in modelos}
proba_oof = {nome: np.full(len(y), np.nan, dtype=np.float32) for nome in modelos}
resultados_fold = []

for fold in folds:
    idx_teste = fold_id == fold
    idx_treino = ~idx_teste
    X_treino, y_treino = X[idx_treino], y[idx_treino]
    X_teste, y_teste = X[idx_teste], y[idx_teste]

    print(f"\nFold {fold}: treino={len(y_treino)}, validação={len(y_teste)}")

    for nome, modelo_base in modelos.items():
        modelo = clone(modelo_base)
        modelo.fit(X_treino, y_treino)
        pred = modelo.predict(X_teste).astype(np.int8)

        # XGBoost e SVM retornam probabilidade da classe positiva (TDAH=1).
        classes_modelo = list(modelo.classes_) if hasattr(modelo, "classes_") else list(modelo.named_steps["clf"].classes_)
        if 1 not in classes_modelo:
            raise ValueError(f"Classe positiva 1 não encontrada no modelo {nome}.")
        indice_positivo = classes_modelo.index(1)
        probas = modelo.predict_proba(X_teste)[:, indice_positivo]

        pred_oof[nome][idx_teste] = pred
        proba_oof[nome][idx_teste] = probas.astype(np.float32)

        tn, fp, fn, tp = confusion_matrix(y_teste, pred, labels=[0, 1]).ravel()
        resultados_fold.append({
            "modelo": nome,
            "fold": int(fold),
            "n_validacao": int(len(y_teste)),
            "accuracy": accuracy_score(y_teste, pred),
            "balanced_accuracy": balanced_accuracy_score(y_teste, pred),
            "precision": precision_score(y_teste, pred, zero_division=0),
            "recall": recall_score(y_teste, pred, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "f1": f1_score(y_teste, pred, zero_division=0),
            "auc_roc": roc_auc_score(y_teste, probas)
        })
        print(
            f"  {nome}: Accuracy={accuracy_score(y_teste, pred):.3f} | "
            f"F1={f1_score(y_teste, pred, zero_division=0):.3f} | "
            f"AUC-ROC={roc_auc_score(y_teste, probas):.3f}"
        )

# Garantir que todos os participantes tenham uma previsão OOF por modelo.
for nome in modelos:
    if np.any(pred_oof[nome] < 0) or not np.isfinite(proba_oof[nome]).all():
        raise RuntimeError(f"Existem previsões OOF ausentes para {nome}.")

res_fold = pd.DataFrame(resultados_fold)
display(res_fold.sort_values(["modelo", "fold"]).round(4))



Fold 0: treino=590, validação=119


/home/maria/TCC_TDAH/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  SVM_RBF: Accuracy=0.555 | F1=0.131 | AUC-ROC=0.449
  Random_Forest: Accuracy=0.706 | F1=0.000 | AUC-ROC=0.495
  XGBoost: Accuracy=0.605 | F1=0.299 | AUC-ROC=0.525

Fold 1: treino=587, validação=122


/home/maria/TCC_TDAH/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  SVM_RBF: Accuracy=0.639 | F1=0.312 | AUC-ROC=0.512
  Random_Forest: Accuracy=0.664 | F1=0.000 | AUC-ROC=0.555
  XGBoost: Accuracy=0.639 | F1=0.083 | AUC-ROC=0.470

Fold 2: treino=580, validação=129


...

/home/maria/TCC_TDAH/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  SVM_RBF: Accuracy=0.624 | F1=0.243 | AUC-ROC=0.550
  Random_Forest: Accuracy=0.611 | F1=0.000 | AUC-ROC=0.578
  XGBoost: Accuracy=0.584 | F1=0.031 | AUC-ROC=0.497

Fold 4: treino=519, validação=190


/home/maria/TCC_TDAH/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  SVM_RBF: Accuracy=0.505 | F1=0.254 | AUC-ROC=0.568
  Random_Forest: Accuracy=0.458 | F1=0.000 | AUC-ROC=0.576
  XGBoost: Accuracy=0.447 | F1=0.037 | AUC-ROC=0.492


,modelo,fold,n_validacao,accuracy,balanced_accuracy,precision,recall,specificity,f1,auc_roc
1,Random_Forest,0,119,0.7059,0.4941,0.0000,0.0000,0.9882,0.0000,0.4946
4,Random_Forest,1,122,0.6639,0.5000,0.0000,0.0000,1.0000,0.0000,0.5548
7,Random_Forest,2,129,0.8372,0.4954,0.0000,0.0000,0.9908,0.0000,0.4842
10,Random_Forest,3,149,0.6107,0.5000,0.0000,0.0000,1.0000,0.0000,0.5784
13,Random_Forest,4,190,0.4579,0.5000,0.0000,0.0000,1.0000,0.0000,0.5765
0,SVM_RBF,0,119,0.5546,0.4235,0.1481,0.1176,0.7294,0.1311,0.4488
3,SVM_RBF,1,122,0.6393,0.5417,0.4348,0.2439,0.8395,0.3125,0.5123
6,SVM_RBF,2,129,0.6047,0.3782,0.0303,0.0500,0.7064,0.0377,0.4312
9,SVM_RBF,3,149,0.6242,0.5391,0.5625,0.1552,0.9231,0.2432,0.5500
12,SVM_RBF,4,190,0.5053,0.5374,0.6957,0.1553,0.9195,0.2540,0.5678


In [7]:
# 5. Resumo por fold e métricas agregadas das previsões OOF
metricas = [
    "accuracy", "balanced_accuracy", "precision",
    "recall", "specificity", "f1", "auc_roc"
]
resumo_cv = (
    res_fold.groupby("modelo")[metricas]
    .agg(["mean", "std"])
    .round(4)
)
display(resumo_cv)

linhas_oof = []
for nome in modelos:
    pred = pred_oof[nome]
    probas = proba_oof[nome]
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    linhas_oof.append({
        "modelo": nome,
        "accuracy_oof": accuracy_score(y, pred),
        "balanced_accuracy_oof": balanced_accuracy_score(y, pred),
        "precision_oof": precision_score(y, pred, zero_division=0),
        "recall_oof": recall_score(y, pred, zero_division=0),
        "specificity_oof": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1_oof": f1_score(y, pred, zero_division=0),
        "auc_roc_oof": roc_auc_score(y, probas),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    })

res_oof = pd.DataFrame(linhas_oof).round(4)
display(res_oof)

print(
    "Atenção: as métricas agregadas OOF resumem as previsões de validação cruzada "
    "agrupada por centro; não representam um teste final independente."
)


accuracy         balanced_accuracy         precision          \
                  mean     std              mean     std      mean     std   
modelo                                                                       
Random_Forest   0.6551  0.1385            0.4979  0.0029    0.0000  0.0000   
SVM_RBF         0.5856  0.0551            0.4840  0.0776    0.3743  0.2792   
XGBoost         0.5776  0.0755            0.4794  0.0332    0.2344  0.1055   

               recall         specificity              f1         auc_roc  \
                 mean     std        mean     std    mean     std    mean   
modelo                                                                      
Random_Forest  0.0000  0.0000      0.9958  0.0058  0.0000  0.0000  0.5377   
SVM_RBF        0.1444  0.0703      0.8236  0.1024  0.1957  0.1100  0.5020   
XGBoost        0.1059  0.1183      0.8528  0.1279  0.1114  0.1093  0.4769   

                       
                  std  
modelo                 
Random_Forest  0.0452  
SVM_RBF        0.0604  
XGBoost        0.0472

,modelo,accuracy_oof,balanced_accuracy_oof,precision_oof,recall_oof,specificity_oof,f1_oof,auc_roc_oof,tn,fp,fn,tp
0,SVM_RBF,0.5797,0.4876,0.3279,0.1562,0.8190,0.2116,0.4532,371,82,216,40
1,Random_Forest,0.6361,0.4978,0.0000,0.0000,0.9956,0.0000,0.4212,451,2,256,0
2,XGBoost,0.5656,0.4579,0.2045,0.0703,0.8455,0.1047,0.4058,383,70,238,18


Atenção: as métricas agregadas OOF resumem as previsões de validação cruzada agrupada por centro; não representam um teste final independente.


In [8]:
# 6. Salvar tabelas, previsões OOF e modelos finais
# Resultados: prioriza o nome definido no config.py; usa aliases para compatibilidade.
pasta_resultados = Path(
    getattr(config, "RESULTS", getattr(config, "RESULTADOS", raiz_projeto / "resultados"))
)
pasta_resultados.mkdir(parents=True, exist_ok=True)

res_fold.to_csv(pasta_resultados / "08_baselines_por_fold.csv", index=False)
resumo_cv.to_csv(pasta_resultados / "08_baselines_resumo_cv.csv")
res_oof.to_csv(pasta_resultados / "08_baselines_metricas_oof.csv", index=False)

arrays_salvar = {
    "y": y,
    "ids": ids,
    "sites": sites,
    "fold_id": fold_id
}
for nome in modelos:
    chave = nome.lower()
    arrays_salvar[f"pred_{chave}"] = pred_oof[nome]
    arrays_salvar[f"proba_{chave}"] = proba_oof[nome]
np.savez_compressed(
    pasta_resultados / "08_baselines_predicoes_oof.npz",
    **arrays_salvar
)

# Salvar estimadores ajustados em todos os dados apenas para uso posterior/inspeção.
# Esses modelos NÃO são usados para calcular as métricas OOF acima.
import joblib
modelos_finais = {}
for nome, modelo_base in modelos.items():
    modelo_final = clone(modelo_base)
    modelo_final.fit(X, y)
    modelos_finais[nome] = modelo_final

pasta_modelos = Path(
    getattr(config, "MODELS", getattr(config, "MODELOS", raiz_projeto / "modelos"))
)
pasta_modelos.mkdir(parents=True, exist_ok=True)
joblib.dump(modelos_finais, pasta_modelos / "08_modelos_baselines_svm_rf_xgboost.joblib")

print("Resultados salvos em:", pasta_resultados)
print("Modelos finais salvos em:", pasta_modelos / "08_modelos_baselines_svm_rf_xgboost.joblib")
print("Arquivos gerados:")
for nome in [
    "08_baselines_por_fold.csv",
    "08_baselines_resumo_cv.csv",
    "08_baselines_metricas_oof.csv",
    "08_baselines_predicoes_oof.npz"
]:
    print(" -", pasta_resultados / nome)


/home/maria/TCC_TDAH/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Resultados salvos em: /home/maria/TCC_TDAH/results
Modelos finais salvos em: /home/maria/TCC_TDAH/models/08_modelos_baselines_svm_rf_xgboost.joblib
Arquivos gerados:
 - /home/maria/TCC_TDAH/results/08_baselines_por_fold.csv
 - /home/maria/TCC_TDAH/results/08_baselines_resumo_cv.csv
 - /home/maria/TCC_TDAH/results/08_baselines_metricas_oof.csv
 - /home/maria/TCC_TDAH/results/08_baselines_predicoes_oof.npz
